In [1]:
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn joblib requests

In [2]:
import os
import glob
import zipfile
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
    explained_variance_score
)

from xgboost import XGBRegressor

In [3]:
# download dataset
# Kelmarsh Wind Farm SCADA data (Plumley / Cubico Sustainable Investments, 2022)
# Zenodo record 7212475, CC-BY-4.0: https://zenodo.org/records/7212475

ZENODO_RECORD_ID = "7212475"
DATA_DIR = "kelmarsh_data"
os.makedirs(DATA_DIR, exist_ok=True)

record = requests.get(f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}", timeout=60).json()

# the record also bundles Grid/PMU/MERRA2/ERA5/kmz files we don't need -
# only keep the yearly 10-minute SCADA zips
scada_files = [f for f in record["files"] if "SCADA" in f["key"] and f["key"].endswith(".zip")]

print("Available SCADA files on Zenodo:")
for f in scada_files:
    print(f" - {f['key']}  ({f['size'] / 1e6:.1f} MB)")

# keep the notebook lightweight: use the two most recent years available
YEARS_TO_USE = ["2020", "2021"]
selected_files = [f for f in scada_files if any(y in f["key"] for y in YEARS_TO_USE)]

for f in selected_files:
    zip_path = os.path.join(DATA_DIR, f["key"])
    if not os.path.exists(zip_path):
        print(f"\nDownloading {f['key']} ...")
        r = requests.get(f["links"]["self"], timeout=600)
        r.raise_for_status()
        with open(zip_path, "wb") as out:
            out.write(r.content)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)

print("\nExtracted files:")
for file in glob.glob(DATA_DIR + "/**/*", recursive=True):
    print(file)

Available SCADA files on Zenodo:
 - Kelmarsh_SCADA_2017_3083.zip  (174.6 MB)
 - Kelmarsh_SCADA_2020_3086.zip  (473.7 MB)
 - Kelmarsh_SCADA_2019_3085.zip  (311.3 MB)
 - Kelmarsh_SCADA_2018_3084.zip  (262.2 MB)
 - Kelmarsh_SCADA_2021_3087.zip  (225.2 MB)
 - Kelmarsh_SCADA_2016_3082.zip  (97.9 MB)

Extracted files:
kelmarsh_data/Status_Kelmarsh_6_2021-01-01_-_2021-07-01_233.csv
kelmarsh_data/Turbine_Data_Kelmarsh_3_2020-01-01_-_2021-01-01_230.csv
kelmarsh_data/Kelmarsh_SCADA_2021_3087.zip
kelmarsh_data/Status_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv
kelmarsh_data/Status_Kelmarsh_1_2021-01-01_-_2021-07-01_228.csv
kelmarsh_data/Status_Kelmarsh_3_2021-01-01_-_2021-07-01_230.csv
kelmarsh_data/Status_Kelmarsh_5_2020-01-01_-_2021-01-01_232.csv
kelmarsh_data/Status_Kelmarsh_6_2020-01-01_-_2021-01-01_233.csv
kelmarsh_data/Status_Kelmarsh_4_2021-01-01_-_2021-07-01_231.csv
kelmarsh_data/Status_Kelmarsh_2_2020-01-01_-_2021-01-01_229.csv
kelmarsh_data/Status_Kelmarsh_2_2021-01-01_-_2021-07-01_229.c

In [4]:
# loading the dataset
# each yearly zip contains one 10-minute SCADA csv per turbine, plus a
# status/events log csv per turbine - we only want the numeric SCADA files
csv_files = [
    f for f in glob.glob(DATA_DIR + "/**/*.csv", recursive=True)
    if "Status" not in os.path.basename(f)
]

print(f"Found {len(csv_files)} turbine SCADA files")
for f in csv_files:
    print(f)

Found 12 turbine SCADA files
kelmarsh_data/Turbine_Data_Kelmarsh_3_2020-01-01_-_2021-01-01_230.csv
kelmarsh_data/Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv
kelmarsh_data/Turbine_Data_Kelmarsh_1_2021-01-01_-_2021-07-01_228.csv
kelmarsh_data/Turbine_Data_Kelmarsh_5_2021-01-01_-_2021-07-01_232.csv
kelmarsh_data/Turbine_Data_Kelmarsh_3_2021-01-01_-_2021-07-01_230.csv
kelmarsh_data/Turbine_Data_Kelmarsh_4_2020-01-01_-_2021-01-01_231.csv
kelmarsh_data/Turbine_Data_Kelmarsh_6_2020-01-01_-_2021-01-01_233.csv
kelmarsh_data/Turbine_Data_Kelmarsh_2_2021-01-01_-_2021-07-01_229.csv
kelmarsh_data/Turbine_Data_Kelmarsh_2_2020-01-01_-_2021-01-01_229.csv
kelmarsh_data/Turbine_Data_Kelmarsh_6_2021-01-01_-_2021-07-01_233.csv
kelmarsh_data/Turbine_Data_Kelmarsh_4_2021-01-01_-_2021-07-01_231.csv
kelmarsh_data/Turbine_Data_Kelmarsh_5_2020-01-01_-_2021-01-01_232.csv


In [5]:
def load_scada_csv(path):
    # the raw Kelmarsh files start with a short metadata preamble
    # (turbine name, stated units, etc.) before the real header row
    header_row = 9
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        for i, line in enumerate(fh):
            if "Date and time" in line:
                header_row = i
                break

    turbine_df = pd.read_csv(path, skiprows=header_row, low_memory=False)
    turbine_df.columns = [c.strip().lstrip("#").strip() for c in turbine_df.columns]

    # turbine number is embedded in the filename, e.g. Turbine_Data_Kelmarsh_1_...
    fname = os.path.basename(path)
    turbine_id = fname.split("Kelmarsh_")[1].split("_")[0] if "Kelmarsh_" in fname else "unknown"
    turbine_df["Turbine"] = turbine_id

    return turbine_df

df = pd.concat([load_scada_csv(f) for f in csv_files], ignore_index=True)

print("Shape:", df.shape)
display(df.head())

Shape: (472608, 300)


,Date and time,Wind speed (m/s),"Wind speed, Standard deviation (m/s)","Wind speed, Minimum (m/s)","Wind speed, Maximum (m/s)",Long Term Wind (m/s),Wind speed Sensor 1 (m/s),"Wind speed Sensor 1, Standard deviation (m/s)","Wind speed Sensor 1, Minimum (m/s)","Wind speed Sensor 1, Maximum (m/s)",...,"Tower Acceleration X, Min (mm/ss)","Tower Acceleration X, Max (mm/ss)","Tower Acceleration Y, Min (mm/ss)","Tower Acceleration Y, Max (mm/ss)","Drive train acceleration, Max (mm/ss)","Drive train acceleration, Min (mm/ss)","Drive train acceleration, StdDev (mm/ss)","Tower Acceleration X, StdDev (mm/ss)","Tower Acceleration Y, StdDev (mm/ss)",Turbine
0,2020-01-01 00:00:00,4.772166,1.164211,3.066588,7.442883,7.1,4.864278,1.103869,3.337130,7.376515,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
1,2020-01-01 00:10:00,4.159083,0.840394,2.615519,5.740701,7.1,4.567990,0.931068,2.872747,6.468347,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
2,2020-01-01 00:20:00,3.917919,0.643606,2.730606,5.013052,7.1,4.139776,0.798626,2.736536,5.464886,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
3,2020-01-01 00:30:00,3.705453,0.472179,3.146406,4.763200,7.1,3.986812,0.560475,2.996876,5.012587,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
4,2020-01-01 00:40:00,3.613699,0.643238,2.628884,4.680782,7.1,3.895515,0.637470,2.754661,4.968647,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3


In [ ]:
#inspecting the dataset

print(df.info())

In [ ]:
print(df.isnull().sum().sort_values(ascending=False).head(20))

In [8]:
# identify the timestamp and power output columns (exact wording can vary
# slightly between dataset versions, so match on keywords rather than hardcoding)
DATETIME_COL = [c for c in df.columns if "date" in c.lower() and "time" in c.lower()][0]
POWER_COL = [c for c in df.columns if "power" in c.lower() and "kw" in c.lower() and "theoretical" not in c.lower()][0]

print("Timestamp column:", DATETIME_COL)
print("Target column:", POWER_COL)
print(df[POWER_COL].describe())

Timestamp column: Date and time
Target column: Power (kW)
count    469463.000000
mean        650.290225
std         669.195083
min         -17.793049
25%         101.689954
50%         387.867709
75%        1057.359720
max        2082.639758
Name: Power (kW), dtype: float64


In [9]:
#separate x and y, engineer simple time-based features, clean target

df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL], errors="coerce")
df = df.dropna(subset=[DATETIME_COL, POWER_COL])

df["hour"] = df[DATETIME_COL].dt.hour
df["month"] = df[DATETIME_COL].dt.month
df["dayofyear"] = df[DATETIME_COL].dt.dayofyear

# keep numeric SCADA signals (avg/min/max/stdev of wind speed, rotor speed,
# blade pitch, temperatures, etc.) as features, plus engineered time features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c != POWER_COL]

X = df[feature_cols + ["Turbine"]].copy()
y = df[POWER_COL].astype("float32")

X = pd.get_dummies(X, columns=["Turbine"], prefix="Turbine")
X = X.fillna(X.median(numeric_only=True))
X = X.astype("float32")

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (469463, 306)
y shape: (469463,)


In [10]:
#splitting dataset into training(72%), validation(8%) and testing(20%)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.10,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (338013, 306)
Validation: (37557, 306)
Testing: (93893, 306)


In [ ]:
#model 1: random forest

rf_model = RandomForestRegressor(
    n_estimators=250,
    max_depth=20,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest training completed.")

In [ ]:
# evaluating model 1 on validation set

rf_val_pred = rf_model.predict(X_val)

rf_val_mae = mean_absolute_error(y_val, rf_val_pred)
rf_val_rmse = np.sqrt(mean_squared_error(y_val, rf_val_pred))
rf_val_r2 = r2_score(y_val, rf_val_pred)
rf_val_mape = mean_absolute_percentage_error(y_val, rf_val_pred)

print(f"MAE:  {rf_val_mae:.2f} kW")
print(f"RMSE: {rf_val_rmse:.2f} kW")
print(f"R2:   {rf_val_r2:.4f}")
print(f"MAPE: {rf_val_mape:.4f}")

In [ ]:
# model 2: XGBoost

xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=8,
    learning_rate=0.08,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_lambda=2.0,
    reg_alpha=0.0,
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=30,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("XGBoost training completed.")

In [ ]:
# evaluating model 2 on validation set

xgb_val_pred = xgb_model.predict(X_val)

xgb_val_mae = mean_absolute_error(y_val, xgb_val_pred)
xgb_val_rmse = np.sqrt(mean_squared_error(y_val, xgb_val_pred))
xgb_val_r2 = r2_score(y_val, xgb_val_pred)
xgb_val_mape = mean_absolute_percentage_error(y_val, xgb_val_pred)

print(f"MAE:  {xgb_val_mae:.2f} kW")
print(f"RMSE: {xgb_val_rmse:.2f} kW")
print(f"R2:   {xgb_val_r2:.4f}")
print(f"MAPE: {xgb_val_mape:.4f}")

In [ ]:
# comparing both models on validation

validation_results = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "MAE": [rf_val_mae, xgb_val_mae],
    "RMSE": [rf_val_rmse, xgb_val_rmse],
    "R2": [rf_val_r2, xgb_val_r2],
    "MAPE": [rf_val_mape, xgb_val_mape]
})

display(validation_results)

In [ ]:
# evaluating models on testing data

rf_test_pred = rf_model.predict(X_test)
xgb_test_pred = xgb_model.predict(X_test)

In [ ]:
# calculating various metrics

def calculate_metrics(model_name, y_true, y_pred):
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
        "Explained Variance": explained_variance_score(y_true, y_pred)
    }

test_results = pd.DataFrame([
    calculate_metrics("Random Forest", y_test, rf_test_pred),
    calculate_metrics("XGBoost", y_test, xgb_test_pred)
])

display(test_results)

In [ ]:
# predicted vs actual power output: random forest

plt.figure(figsize=(8, 7))

plt.scatter(y_test, rf_test_pred, s=4, alpha=0.15, color="tab:blue")
lims = [0, max(y_test.max(), rf_test_pred.max())]
plt.plot(lims, lims, "r--", linewidth=1.5)

plt.xlabel("Actual power (kW)")
plt.ylabel("Predicted power (kW)")
plt.title("Random Forest: Predicted vs Actual Power Output")
plt.show()

In [ ]:
# predicted vs actual power output: XGBoost

plt.figure(figsize=(8, 7))

plt.scatter(y_test, xgb_test_pred, s=4, alpha=0.15, color="tab:green")
lims = [0, max(y_test.max(), xgb_test_pred.max())]
plt.plot(lims, lims, "r--", linewidth=1.5)

plt.xlabel("Actual power (kW)")
plt.ylabel("Predicted power (kW)")
plt.title("XGBoost: Predicted vs Actual Power Output")
plt.show()

In [ ]:
# residual distribution for both models

rf_residuals = y_test.values - rf_test_pred
xgb_residuals = y_test.values - xgb_test_pred

plt.figure(figsize=(10, 6))
sns.kdeplot(rf_residuals, label="Random Forest", fill=True, alpha=0.3)
sns.kdeplot(xgb_residuals, label="XGBoost", fill=True, alpha=0.3)

plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("Residual (Actual - Predicted, kW)")
plt.title("Residual Distribution")
plt.legend()
plt.show()

In [ ]:
# feature importance (top 15) - useful for maintenance / anomaly-flagging insight

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

rf_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
xgb_importance = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

sns.barplot(x=rf_importance.values, y=rf_importance.index, ax=axes[0], color="tab:blue")
axes[0].set_title("Random Forest: Top 15 Features")

sns.barplot(x=xgb_importance.values, y=xgb_importance.index, ax=axes[1], color="tab:green")
axes[1].set_title("XGBoost: Top 15 Features")

plt.tight_layout()
plt.show()

In [ ]:
# overfitting check

rf_train_pred = rf_model.predict(X_train)
xgb_train_pred = xgb_model.predict(X_train)

rf_train_r2 = r2_score(y_train, rf_train_pred)
rf_test_r2 = r2_score(y_test, rf_test_pred)

xgb_train_r2 = r2_score(y_train, xgb_train_pred)
xgb_test_r2 = r2_score(y_test, xgb_test_pred)

print(f"Random Forest - Train R2: {rf_train_r2:.4f} | Test R2: {rf_test_r2:.4f}")
print(f"XGBoost       - Train R2: {xgb_train_r2:.4f} | Test R2: {xgb_test_r2:.4f}")

In [ ]:
plot_df = pd.DataFrame({
    "Model": ["Random Forest", "Random Forest", "XGBoost", "XGBoost"],
    "Dataset": ["Training", "Testing", "Training", "Testing"],
    "R2": [
        rf_train_r2,
        rf_test_r2,
        xgb_train_r2,
        xgb_test_r2
    ]
})

plt.figure(figsize=(9, 6))

sns.barplot(
    data=plot_df,
    x="Model",
    y="R2",
    hue="Dataset"
)

plt.ylim(0, 1)
plt.title("Training vs Testing R2 (Overfitting Check)")
plt.show()

In [ ]:
if xgb_val_r2 >= rf_val_r2:
    best_model = xgb_model
    best_model_name = "XGBoost"
else:
    best_model = rf_model
    best_model_name = "Random Forest"

print("Selected model:", best_model_name)

In [ ]:
model_bundle = {
    "model": best_model,
    "feature_columns": list(X.columns),
    "target_column": POWER_COL,
    "model_name": best_model_name
}

joblib.dump(
    model_bundle,
    "kelmarsh_power_model.pkl"
)

print("Model saved successfully.")

In [27]:
#download as .pkl file

import os

file_size = os.path.getsize("kelmarsh_power_model.pkl")

print("File:", "kelmarsh_power_model.pkl")
print("Size:", round(file_size / (1024 * 1024), 2), "MB")

from google.colab import files

files.download("kelmarsh_power_model.pkl")

File: kelmarsh_power_model.pkl
Size: 1.45 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>